# 06 -- Spark MLlib

The last module in the tool progression: pandas -> scikit-learn ->
HuggingFace -> PyTorch/TensorFlow -> Spark MLlib. 230 rows is absurdly
small for Spark -- the point isn't performance, it's the API. This
notebook reimplements Notebook 02's feature engineering with the Spark
DataFrame API instead of pandas (`src/vfr/features.py` was written
dependency-light specifically so its feature *definitions* could be
reimplemented here without importing pandas-specific code), then trains
Spark MLlib's `GBTRegressor` -- the same model family that won Notebook
03 -- through an MLlib `Pipeline` + `CrossValidator`.

Runs inside the Docker container -- needs a JVM (`default-jdk-headless`,
added to the Dockerfile) that `sentence-transformers`/`torch`/`tensorflow`
didn't.

**2026-08-30 note:** the hardcoded comparison numbers in this notebook (GradientBoosting MAE 1.189, etc.) are from Notebook 03's satellite-imagery-based labeling pass, since replaced by FAA VFR sectional chart labeling (see data/labels/chart_picks.csv). They'll need refreshing once the route is relabeled and Notebook 03 is rerun -- rerun this notebook after that to get honest numbers.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

spark = SparkSession.builder.appName("vfr_route_mllib").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

RANDOM_STATE = 42
EARTH_RADIUS_NM = 3440.065  # src/vfr/geo.py -- same constant, so nn_dist_nm matches Notebook 02


## Step 1 -- Load the raw candidates table

This is Notebook 01's output, *before* Notebook 02's pandas feature
engineering -- `bbox_area_m2`, route geometry (`cross_track_nm`,
`along_track_nm`), and the raw `name`/`category`/`lat`/`lon` are already
there; `log_size`, `name_uniqueness`, `nn_dist_nm`, and the one-hot
`category_*` columns are not -- those get rebuilt below, in Spark.

In [ ]:
candidates_sdf = (
    spark.read.csv(
        str(PROJECT_ROOT / "data" / "processed" / "candidates_c81_kdlh.csv"),
        header=True, inferSchema=True, quote='"', escape='"',
        # escape must match quote here -- pandas wrote this CSV with RFC4180-style
        # doubled-quote escaping (""...""), but Spark defaults to backslash escaping,
        # which silently misparses the quoted `tags` JSON column and shifts every
        # column after it (cross_track_nm etc. end up as string fragments of tags).
    )
    .select("osm_id", "osm_type", "category", "name", "lat", "lon", "bbox_area_m2",
            "cross_track_nm", "along_track_nm", "within_preferred_corridor")
)

print(f"{candidates_sdf.count()} candidates")
candidates_sdf.printSchema()


## Step 2 -- Feature engineering, the Spark way

**`log_size`** -- `log1p(bbox_area_m2)`, identical formula to
`features.log_size_feature`, just `pyspark.sql.functions.log1p` instead of
`numpy.log1p`.

**`name_uniqueness`** -- pandas used `value_counts()` + `.map()`; the
Spark equivalent is a window function: count rows sharing each `name`,
then `1/count`, null propagates automatically for unnamed candidates
(no need for an explicit NaN check).

In [ ]:
candidates_sdf = candidates_sdf.withColumn("log_size", F.log1p(F.col("bbox_area_m2")))

name_window = Window.partitionBy("name")
candidates_sdf = candidates_sdf.withColumn(
    "name_uniqueness",
    F.when(F.col("name").isNull(), None).otherwise(1.0 / F.count("name").over(name_window)),
)

candidates_sdf.select("name", "log_size", "name_uniqueness").show(5, truncate=False)


**`nn_dist_nm`** -- the pandas version built a full 230x230 pairwise
haversine matrix with numpy broadcasting. There's no dense-matrix
equivalent in Spark DataFrames -- the DataFrame way is a **self cross
join**, computing haversine distance per pair, then `groupBy` +
`min` to keep only each candidate's closest neighbor. 230 candidates ->
230x229 ~= 52,670 pairs, trivial for Spark, but this is the shape a real
"nearest neighbor" join takes at any scale in this API, which is the
actual point of doing it here.

In [ ]:
left = candidates_sdf.select(
    F.col("osm_id").alias("l_osm_id"), F.col("osm_type").alias("l_osm_type"),
    F.radians(F.col("lat")).alias("l_lat_r"), F.radians(F.col("lon")).alias("l_lon_r"),
)
right = candidates_sdf.select(
    F.col("osm_id").alias("r_osm_id"), F.col("osm_type").alias("r_osm_type"),
    F.radians(F.col("lat")).alias("r_lat_r"), F.radians(F.col("lon")).alias("r_lon_r"),
)

pairs = left.crossJoin(right).where(
    (F.col("l_osm_id") != F.col("r_osm_id")) | (F.col("l_osm_type") != F.col("r_osm_type"))
)

dlat = F.col("r_lat_r") - F.col("l_lat_r")
dlon = F.col("r_lon_r") - F.col("l_lon_r")
a = F.sin(dlat / 2) ** 2 + F.cos(F.col("l_lat_r")) * F.cos(F.col("r_lat_r")) * F.sin(dlon / 2) ** 2
haversine_nm = 2 * EARTH_RADIUS_NM * F.asin(F.sqrt(a))

pairs = pairs.withColumn("dist_nm", haversine_nm)

nn_dist = (
    pairs.groupBy("l_osm_id", "l_osm_type")
    .agg(F.min("dist_nm").alias("nn_dist_nm"))
    .withColumnRenamed("l_osm_id", "osm_id")
    .withColumnRenamed("l_osm_type", "osm_type")
)

candidates_sdf = candidates_sdf.join(nn_dist, on=["osm_id", "osm_type"], how="left")

candidates_sdf.select("name", "nn_dist_nm").show(5, truncate=False)


In [ ]:
# Sanity check against Notebook 02's pandas output (same formula, same constant):
# nn_dist_nm mean ~0.854, log_size mean ~9.464 over all 230 candidates.
candidates_sdf.select(F.mean("nn_dist_nm"), F.mean("log_size")).show()


**Category one-hot** -- `StringIndexer` (category string -> integer
code) feeding `OneHotEncoder` (integer code -> sparse vector) is MLlib's
idiomatic two-stage encoding, not a `pd.get_dummies()`-style manual column
explosion. It also plugs directly into the `Pipeline` in Step 5 instead
of needing separate preprocessing.

In [ ]:
category_indexer = StringIndexer(inputCol="category", outputCol="category_idx")
category_encoder = OneHotEncoder(inputCol="category_idx", outputCol="category_vec")


## Step 3 -- Join with labels

Same 228 labeled candidates as every prior notebook.

In [ ]:
labels_sdf = spark.read.csv(
    str(PROJECT_ROOT / "data" / "labels" / "spottability_ratings.csv"), header=True, inferSchema=True
).select("osm_id", "osm_type", F.col("rating").cast("double"))

labeled_sdf = candidates_sdf.join(labels_sdf, on=["osm_id", "osm_type"]).fillna({"name_uniqueness": 0.0})

print(f"{labeled_sdf.count()} labeled candidates")


## Step 4 -- Stratified train/test split

`train_test_split(..., stratify=y)` has no direct DataFrame equivalent --
the Spark way is `sampleBy`, which takes a per-class sampling fraction (0.8
for every rating here, an 80/20 split within each rating value) and a
seed. It only returns one side, so the test set is the anti-join
complement rather than a second `sampleBy` call -- `sampleBy` on its own
doesn't guarantee a clean partition otherwise.

In [ ]:
train_sdf = labeled_sdf.sampleBy("rating", fractions={r: 0.8 for r in [1.0, 2.0, 3.0, 4.0, 5.0]}, seed=RANDOM_STATE)
test_sdf = labeled_sdf.join(train_sdf.select("osm_id", "osm_type"), on=["osm_id", "osm_type"], how="left_anti")

print(f"train: {train_sdf.count()}, test: {test_sdf.count()}")
train_sdf.groupBy("rating").count().orderBy("rating").show()
test_sdf.groupBy("rating").count().orderBy("rating").show()


## Step 5 -- Dummy baseline

No `DummyRegressor` in MLlib -- "predict the training mean" is just an
aggregation plus a manual MAE calculation.

In [ ]:
train_mean_rating = train_sdf.select(F.mean("rating")).first()[0]

dummy_mae = (
    test_sdf.withColumn("abs_err", F.abs(F.col("rating") - F.lit(train_mean_rating)))
    .select(F.mean("abs_err"))
    .first()[0]
)
print(f"Dummy (train mean = {train_mean_rating:.3f}) -- held-out MAE: {dummy_mae:.3f}")


## Step 6 -- MLlib Pipeline + CrossValidator

`VectorAssembler` combines the numeric features and the one-hot
`category_vec` into a single `features` vector column -- every MLlib
model expects exactly that shape, unlike sklearn's plain 2D array/
DataFrame. The full `Pipeline` (index -> encode -> assemble -> model) gets
handed to `CrossValidator`, MLlib's equivalent of `GridSearchCV`, doing
5-fold CV over a small `GBTRegressor` hyperparameter grid -- same model
family, same tuning spirit as Notebook 03's Step 8, different API.

In [ ]:
NUMERIC_FEATURE_COLS = [
    "cross_track_nm", "along_track_nm", "within_preferred_corridor", "log_size",
    "name_uniqueness", "nn_dist_nm",
]

assembler = VectorAssembler(inputCols=NUMERIC_FEATURE_COLS + ["category_vec"], outputCol="features")

gbt = GBTRegressor(featuresCol="features", labelCol="rating", seed=RANDOM_STATE)

pipeline = Pipeline(stages=[category_indexer, category_encoder, assembler, gbt])

param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [2, 3, 4])
    .addGrid(gbt.maxIter, [20, 50])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .build()
)

evaluator = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="mae")

cv = CrossValidator(
    estimator=pipeline, estimatorParamMaps=param_grid, evaluator=evaluator,
    numFolds=5, seed=RANDOM_STATE, parallelism=4,
)

cv_model = cv.fit(train_sdf)

best_cv_mae = min(cv_model.avgMetrics)
print(f"Best CV MAE: {best_cv_mae:.3f}")


## Step 7 -- Held-out evaluation and comparison

**Caveat before the numbers:** this is *not* the identical row-for-row
split Notebook 04 reused from Notebook 03 -- `sampleBy` and sklearn's
`train_test_split` select different rows even with matching seeds/
fractions, so this is a same-*shape*, different-*rows* held-out set. Take
the comparison below as directional, not a precise head-to-head the way
Notebook 04 vs. 03 was.

In [ ]:
predictions = cv_model.transform(test_sdf)

test_mae = evaluator.evaluate(predictions)
test_rmse = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="rmse").evaluate(predictions)
test_r2 = RegressionEvaluator(labelCol="rating", predictionCol="prediction", metricName="r2").evaluate(predictions)

print(f"Spark GBTRegressor held-out -- MAE: {test_mae:.3f}  RMSE: {test_rmse:.3f}  R^2: {test_r2:.3f}")

print("\nFor comparison, same-shaped held-out numbers from earlier notebooks:")
current_metrics_path = PROJECT_ROOT / "data" / "models" / "current" / "metrics.json"
if current_metrics_path.exists():
    baseline_metrics = json.loads(current_metrics_path.read_text())
    print(
        f"  {baseline_metrics['model_type']} (promoted, pipeline.retrain): "
        f"MAE {baseline_metrics['held_out_mae']:.3f}  "
        f"RMSE {baseline_metrics['held_out_rmse']:.3f}  "
        f"R^2 {baseline_metrics['held_out_r2']:.3f}"
    )
else:
    print("  (no promoted model yet at data/models/current/metrics.json -- run the")
    print("   pipeline's retrain+evaluate+promote for a live baseline; showing the")
    print("   last hand-recorded satellite-labeled Notebook 03 run instead, stale:)")
    print("  GradientBoosting (sklearn, Notebook 03, satellite-labeled, stale): MAE 1.189  RMSE 1.560  R^2 -0.109")
nb04_metrics_path = PROJECT_ROOT / "data" / "models" / "benchmarks" / "04_pytorch_tensorflow.json"
if nb04_metrics_path.exists():
    nb04 = json.loads(nb04_metrics_path.read_text())
    for label, key in [("PyTorch MLP", "pytorch_mlp"), ("Keras MLP", "keras_mlp")]:
        m = nb04[key]
        print(f"  {label} (Notebook 04): MAE {m['MAE']:.3f}  RMSE {m['RMSE']:.3f}  R^2 {m['R2']:.3f}")
else:
    print("  (Notebook 04 hasn't been run/saved its metrics yet -- no PyTorch/Keras comparison)")
print("  (Notebook 03's nested-CV GradientBoosting MAE was 1.021 +/- 0.064 -- the more")
print("   trustworthy number given single-split noise; no exact Spark analog run here.)")


In [ ]:
spark.stop()
